In [ ]:
import requests
from bs4 import BeautifulSoup
import re, json, time, random

N_ARTICLES = 100  # bump this up once both versions check out

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
}

def extract_categories(html):
    """Pull all franchise-type tags (real categories) — excludes person/keyword tags."""
    tag_objs = re.findall(r'\{"headline":"[^{}]*"__typename":"tag"\}', html)
    cats = {}
    for raw in tag_objs:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if obj.get("type") == "franchise":
            cats[obj["id"]] = obj.get("tagName") or obj.get("headline")
    return list(cats.values())

def parse_article(url, html):
    soup = BeautifulSoup(html, "html.parser")

    ld_script = soup.find("script", type="application/ld+json")
    ld = {}
    if ld_script and ld_script.string:
        try:
            ld = json.loads(ld_script.string)
        except json.JSONDecodeError:
            pass

    title = ld.get("headline", "")
    authors = "; ".join(a.get("name", "") for a in ld.get("author", []) if isinstance(a, dict))
    date_published = ld.get("datePublished", "")
    date_modified = ld.get("dateModified", "")

    body_div = soup.find("div", class_="ArticleBody-articleBody")
    content = ""
    if body_div:
        paragraphs = body_div.find_all("p")
        content = "\n".join(p.get_text(" ", strip=True) for p in paragraphs)

    categories = extract_categories(html)

    return {
        "url": url,
        "title": title,
        "author": authors,
        "date_published": date_published,
        "date_modified": date_modified,
        "categories": categories,   # kept as a list here — each version below formats it differently
        "content": content,
    }

def fetch_and_parse(url):
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code != 200:
            return None
        return parse_article(url, resp.text)
    except Exception as e:
        print(f"failed: {url} ({e})")
        return None

In [ ]:
import sqlite3
import pandas as pd

sitemap_df = pd.read_csv("merged_sitemaps.csv")
urls_to_scrape = sitemap_df["loc"].head(N_ARTICLES).tolist()

DB_PATH = "cnbc_articles.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.executescript("""
CREATE TABLE IF NOT EXISTS articles (
    url TEXT PRIMARY KEY,
    title TEXT,
    author TEXT,
    date_published TEXT,
    date_modified TEXT,
    content TEXT
);
CREATE TABLE IF NOT EXISTS article_categories (
    url TEXT,
    category TEXT,
    FOREIGN KEY(url) REFERENCES articles(url)
);
""")
conn.commit()

for i, url in enumerate(urls_to_scrape, 1):
    result = fetch_and_parse(url)
    if result:
        cur.execute(
            "INSERT OR REPLACE INTO articles (url, title, author, date_published, date_modified, content) VALUES (?, ?, ?, ?, ?, ?)",
            (result["url"], result["title"], result["author"], result["date_published"], result["date_modified"], result["content"])
        )
        cur.execute("DELETE FROM article_categories WHERE url = ?", (result["url"],))  # avoid dupes on re-run
        for cat in result["categories"]:
            cur.execute("INSERT INTO article_categories (url, category) VALUES (?, ?)", (result["url"], cat))
        conn.commit()
    print(f"{i}/{len(urls_to_scrape)} done: {url}")
    time.sleep(random.uniform(0.5, 1.5))

print(f"\nSaved to {DB_PATH}")
pd.read_sql("SELECT a.title, a.author, group_concat(c.category, '; ') as categories FROM articles a LEFT JOIN article_categories c ON a.url = c.url GROUP BY a.url", conn)

1/500 done: https://www.cnbc.com/2021/03/08/pentagon-uncertain-of-troop-withdrawal-in-afghanistan-as-deadline-looms-.html
2/500 done: https://www.cnbc.com/2021/03/08/stocks-making-the-biggest-moves-after-the-bell-zoom-video-stitch-fix-invitae.html
3/500 done: https://www.cnbc.com/2021/03/08/cathie-wood-says-she-is-still-bullish-on-tesla-hints-at-a-new-price-target.html
4/500 done: https://www.cnbc.com/2021/03/08/cathie-wood-sees-bitcoin-joining-stocks-and-bonds-as-part-of-the-classic-investor-allocation-model.html
5/500 done: https://www.cnbc.com/2021/03/08/covid-melinda-gates-says-we-could-reach-global-herd-immunity-sometime-in-2022.html
6/500 done: https://www.cnbc.com/2021/03/08/forex-markets-bonds-moves-risk-currencies-and-dollar.html
7/500 done: https://www.cnbc.com/2021/03/08/us-bonds-treasury-yields-rise-after-senate-passes-stimulus-package.html
8/500 done: https://www.cnbc.com/2021/03/07/stock-market-open-to-close-news.html
9/500 done: https://www.cnbc.com/2021/03/08/stocks-mak

,url,title,author,date_published,date_modified,categories,content
0,https://www.cnbc.com/2021/03/08/pentagon-uncer...,Pentagon uncertain of U.S. troop withdrawal in...,Amanda Macias,2021-03-08T20:59:00+0000,2021-03-08T22:29:46+0000,Politics; US: News; Defense; US Top News and A...,WASHINGTON — The Pentagon said Monday that it ...
1,https://www.cnbc.com/2021/03/08/stocks-making-...,Stocks making the biggest moves after the bell...,Rich Mendez,2021-03-08T22:16:12+0000,2021-03-08T22:16:12+0000,Finance; stocks; Market Insider; Economy; Markets,Check out the companies making headlines after...
2,https://www.cnbc.com/2021/03/08/cathie-wood-sa...,Cathie Wood says she is still bullish on Tesla...,Kevin Stankiewicz,2021-03-08T21:05:37+0000,2021-03-08T22:08:02+0000,stocks; Investing; Autos; Closing Bell Parent,NaN
3,https://www.cnbc.com/2021/03/08/cathie-wood-se...,Cathie Wood sees bitcoin joining stocks and bo...,Jesse Pound,2021-03-08T21:35:34+0000,2021-03-08T21:43:18+0000,Investing; Cryptocurrency; Bitcoin; Wall Stree...,Bitcoin and other cryptocurrencies could event...
4,https://www.cnbc.com/2021/03/08/covid-melinda-...,Melinda Gates says we could reach global herd ...,Berkeley Lovelace Jr.,2021-03-08T21:36:09+0000,2021-03-08T21:36:09+0000,Coronavirus; Politics; World News; Biotech and...,Billionaire philanthropist and former tech exe...
...,...,...,...,...,...,...,...
495,https://www.cnbc.com/2021/03/03/pwc-the-pandem...,The pandemic is reversing gender equality prog...,Vicky McKeever,2021-03-03T10:00:54+0000,2021-03-03T10:25:41+0000,Make It - Money; Make It - Life; Make It,NaN
496,https://www.cnbc.com/2021/03/03/prudential-202...,Asia growth drives 4% rise in Prudential 2020 ...,NaN,2021-03-03T09:29:09+0000,2021-03-03T10:23:41+0000,NaN,In this article\nPrudential's operating profit...
497,https://www.cnbc.com/2021/03/02/epic-games-buy...,Fortnite creator Epic Games buys the developer...,Ryan Browne,2021-03-02T19:04:37+0000,2021-03-03T10:07:25+0000,Technology; Finance; Deals & IPOs; Media; Vide...,LONDON — Fortnite developer Epic Games has acq...
498,https://www.cnbc.com/2021/03/03/russia-slams-n...,Russia slams 'hostile' new U.S. sanctions and ...,Holly Ellyatt,2021-03-03T09:09:56+0000,2021-03-03T09:09:56+0000,Europe Economy; World Politics; World Economy;...,Moscow rejected new sanctions imposed on it by...
